# EMT and DP Ph3 GFL state-space validation

This notebook validates state-space extraction and modal analysis using an ideal three-phase source, a series $R$-$L$ grid branch, and an averaged grid-following inverter with an $L$-$C$ filter. Four DPsim realizations are compared:

- EMT variable SSN,
- EMT split SSN with a one-sample controller/network delay,
- DP Ph3 variable SSN, and
- DP Ph3 split SSN.

Build the C++ example, then run it from its CMake output directory so that the relative log paths match `build_dir` below:

```bash
cmake --build build --target EMT_DP_Ph3_GFL_StateSpaceValidation
cd build/dpsim/examples/cxx
./EMT_DP_Ph3_GFL_StateSpaceValidation
# Add the longer disturbed simulations used in Study 3:
./EMT_DP_Ph3_GFL_StateSpaceValidation --time-domain
```

Two independently assembled references are used. The **no-delay reference** discretizes the complete continuous $dq0$ model in one trapezoidal step. The **delay-aware reference** discretizes controller and electrical subsystems separately and couples them through the same one-step bridge-voltage delay as the split implementation. Thus, error to the matching reference evaluates implementation/extraction accuracy, whereas deviation from the no-delay reference quantifies the intentional split-delay effect.

The selected mode is an electrical filter/grid resonance near $1.93$ kHz. In the terminology of Hatziargyriou et al., *Definition and Classification of Power System Stability — Revisited and Extended* (IEEE TPWRS, 2021), it is a **fast-interaction converter-driven stability** mode: fast converter controls interact with filter and network dynamics. It is not subsynchronous torsional interaction because this system contains neither series compensation nor a turbine-generator shaft mode.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# The executable is launched from this CMake build directory, and its relative
# log paths are therefore resolved from the same directory.
build_dir = Path("/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx")
result_dir = build_dir / "logs/EMT_DP_Ph3_GFL_StateSpaceValidation"
required = ["parameters.csv", "eigenvalues.csv", "summary.csv", "participation.csv"]
missing = [name for name in required if not (result_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {result_dir}. Run EMT_DP_Ph3_GFL_StateSpaceValidation first."
    )

parameters = pd.read_csv(result_dir / "parameters.csv")
eigenvalues = pd.read_csv(result_dir / "eigenvalues.csv")
summary = pd.read_csv(result_dir / "summary.csv")
participation = pd.read_csv(result_dir / "participation.csv")
manifest_path = result_dir / "time_domain_manifest.csv"
time_manifest = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()
parameter_table = parameters.set_index("parameter")

def display_numeric_table(frame, precision=6):
    """Display a rounded DataFrame without pandas Styler/Jinja2."""
    shown = frame.copy()
    numeric = shown.select_dtypes(include=[np.number]).columns
    shown.loc[:, numeric] = shown.loc[:, numeric].round(precision)
    display(shown)

model_order = [
    "analytical no delay", "analytical split delay", "EMT variable",
    "EMT split", "DP Ph3 variable", "DP Ph3 split",
]
colors = dict(zip(model_order, plt.get_cmap("tab10").colors))
markers = {
    "analytical no delay": "x", "analytical split delay": "+",
    "EMT variable": "o", "EMT split": "s",
    "DP Ph3 variable": "^", "DP Ph3 split": "v",
}

def rows_for(model, dt_us=1.0, study="time_step_sweep", method=None,
             gain_scale=None):
    rows = eigenvalues[
        (eigenvalues.model == model)
        & np.isclose(eigenvalues.time_step_us, dt_us)
        & (eigenvalues.study == study)
    ]
    if method is not None:
        rows = rows[rows.method == method]
    if gain_scale is not None:
        rows = rows[np.isclose(rows.gain_scale, gain_scale)]
    return rows

def finite_modes(rows):
    keep = np.isfinite(rows.lambda_real) & np.isfinite(rows.lambda_imag)
    return rows.loc[keep].copy()

def modal_method(model):
    return None if model.startswith("analytical") else "one_step"

def nearest_mode_in_z(rows, reference_mode):
    distance = np.hypot(rows.z_real - reference_mode.z_real,
                        rows.z_imag - reference_mode.z_imag)
    return rows.loc[distance.idxmin()]

def nearest_mode_in_lambda(rows, reference_mode):
    distance = np.hypot(rows.lambda_real - reference_mode.lambda_real,
                        rows.lambda_imag - reference_mode.lambda_imag)
    return rows.loc[distance.idxmin()]

def oscillatory_mode(model, dt_us=1.0, study="time_step_sweep",
                     method=None, gain_scale=None, minimum_frequency_hz=500.0):
    if method is None:
        method = modal_method(model)
    modes = finite_modes(rows_for(model, dt_us, study, method, gain_scale))
    modes = modes[modes.lambda_imag > 2*np.pi*minimum_frequency_hz]
    if modes.empty:
        raise ValueError(f"No oscillatory mode found for {model}, {study}, {dt_us} us.")
    return modes.loc[modes.lambda_real.idxmax()]

def oscillatory_mode_indices(model):
    modes = finite_modes(rows_for(model, 1.0, method=modal_method(model)))
    critical = oscillatory_mode(model)
    partner = modes.iloc[np.argmin(
        abs(modes.lambda_real.to_numpy() - critical.lambda_real)
        + abs(modes.lambda_imag.to_numpy() + critical.lambda_imag)
    )]
    return list(dict.fromkeys([int(critical["index"]), int(partner["index"])]))

def state_group(model, state_index, state_name):
    name = state_name.lower().split(".")[-1]
    controller = ["psi", "theta_pll", "phi_pll", "p_filtered",
                  "q_filtered", "phi_d", "phi_q", "gamma_d", "gamma_q"]
    dp_controller = ["psi", "phi_pll", "p_filtered", "q_filtered",
                     "phi_d", "phi_q", "gamma_d", "gamma_q"]
    for label in controller:
        if label == name or name.endswith(label):
            return "psi" if label == "theta_pll" else label
    if "delay" in name: return "v_inv_delay"
    if "gridinductor" in name or "i_line" in name: return "i_line"
    if "vc_" in name or name.startswith("vc"): return "v_c"
    if "if_" in name or name.startswith("if"): return "i_f"
    if model == "DP Ph3 variable":
        local_index = state_index - 6
        if 0 <= local_index < 6: return "v_c"
        if 6 <= local_index < 12: return "i_f"
        if 12 <= local_index < 20: return dp_controller[local_index - 12]
    if model == "DP Ph3 split":
        if state_index < 8: return dp_controller[state_index]
        if state_index < 14: return "v_c"
        if state_index < 20: return "i_f"
        if state_index < 26: return "v_inv_delay"
        return "i_line"
    return state_name

def grouped_participation(model):
    selected = participation[
        (participation.model == model)
        & participation.mode_index.isin(oscillatory_mode_indices(model))
    ].copy()
    selected["group"] = [state_group(model, int(i), n)
                         for i, n in zip(selected.state_index, selected.state_name)]
    per_mode = selected.groupby(["mode_index", "group"])[["p_real", "p_imag"]].sum()
    per_mode["magnitude"] = np.hypot(per_mode.p_real, per_mode.p_imag)
    grouped = per_mode.groupby("group").magnitude.sum().sort_values(ascending=False)
    return grouped / grouped.sum()

def tracked_filter_grid_modes(maximum_time_step_us=100.0):
    """Track the 1.93 kHz pair and decompose matched-reference and delay errors."""
    baseline = oscillatory_mode("analytical no delay")
    records = []
    time_steps = sorted(summary.loc[
        (summary.study == "time_step_sweep")
        & (summary.time_step_us <= maximum_time_step_us), "time_step_us"
    ].unique())
    for dt_us in time_steps:
        references = {}
        for reference_model in ["analytical no delay", "analytical split delay"]:
            candidates = finite_modes(rows_for(reference_model, dt_us))
            candidates = candidates[candidates.lambda_imag > 0]
            references[reference_model] = nearest_mode_in_lambda(candidates, baseline)
        no_delay = references["analytical no delay"]
        delayed = references["analytical split delay"]
        predicted_delay_z = np.hypot(delayed.z_real-no_delay.z_real,
                                     delayed.z_imag-no_delay.z_imag)
        for model in model_order:
            reference_name = "analytical split delay" if "split" in model else "analytical no delay"
            reference = references[reference_name]
            if model.startswith("analytical"):
                mode = references[model]
            else:
                candidates = finite_modes(rows_for(model, dt_us, method="one_step"))
                candidates = candidates[candidates.lambda_imag > 0]
                mode = nearest_mode_in_z(candidates, reference)
            frequency = mode.lambda_imag/(2*np.pi)
            records.append({
                "model": model, "time_step_us": dt_us,
                "lambda_real": mode.lambda_real, "frequency_hz": frequency,
                "z_error_matching_reference": np.hypot(
                    mode.z_real-reference.z_real, mode.z_imag-reference.z_imag),
                "z_deviation_no_delay": np.hypot(
                    mode.z_real-no_delay.z_real, mode.z_imag-no_delay.z_imag),
                "predicted_delay_z": predicted_delay_z,
                "damping_error": abs(mode.lambda_real-reference.lambda_real),
                "frequency_error_hz": abs(mode.lambda_imag-reference.lambda_imag)/(2*np.pi),
                "samples_per_period": 1e6/(dt_us*abs(frequency)),
            })
    return pd.DataFrame(records)

def floquet_filter_grid_modes(tracked):
    records = []
    for model in ["EMT variable", "EMT split"]:
        reference_name = "analytical split delay" if "split" in model else "analytical no delay"
        for dt_us in sorted(tracked.time_step_us.unique()):
            reference = tracked[(tracked.model == reference_name)
                                & np.isclose(tracked.time_step_us, dt_us)].iloc[0]
            candidates = finite_modes(rows_for(model, dt_us, method="monodromy"))
            candidates = candidates[candidates.lambda_imag > 0]
            if candidates.empty: continue
            distance = np.hypot(candidates.lambda_real-reference.lambda_real,
                                candidates.lambda_imag-2*np.pi*reference.frequency_hz)
            mode = candidates.loc[distance.idxmin()]
            records.append({"model": model, "time_step_us": dt_us,
                            "lambda_real": mode.lambda_real,
                            "frequency_hz": mode.lambda_imag/(2*np.pi)})
    return pd.DataFrame(records)


## Study 0 — parameters and operating-point check

**Why this is included.** The electrical parameters should describe a recognizable converter/grid interaction, and the disturbed simulations must start from the intended equilibrium. The derived LCL resonance gives a physical interpretation to the selected modal frequency. Before the scheduled disturbance, relative $P/Q$ errors verify that the subsequent oscillation is not an initialization transient.


In [ ]:
parameter_names = [
    "frequency", "voltage_rms_ll", "active_power_reference",
    "reactive_power_reference", "grid_resistance", "grid_inductance",
    "filter_inductance", "filter_capacitance", "filter_resistance",
    "coupling_resistance", "grid_x_over_r", "operating_point_scr",
    "undamped_lcl_resonance_frequency", "stable_gain_scale",
    "unstable_gain_scale", "perturbation_time", "perturbation_duration",
    "grid_voltage_pulse_relative",
]
display(parameters[parameters.parameter.isin(parameter_names)].set_index("parameter"))

if not time_manifest.empty:
    equilibrium = time_manifest[[
        "model", "stability_case", "time_step_us",
        "pre_perturbation_p_error", "pre_perturbation_q_error",
    ]].copy()
    display(equilibrium.sort_values(["stability_case", "time_step_us", "model"]))
else:
    print("Run with --time-domain to generate the pre-disturbance equilibrium check.")


**Expected conclusion.** The undamped LCL resonance should be close to the extracted $1.93$ kHz branch. Small pre-disturbance $P/Q$ errors show that the model remains at its initialized operating point for one complete fundamental period before the pulse is applied.

## Study 1 — small-step validation ($\Delta t=1\,\mu$s)

**Why this is included.** At $1\,\mu$s, the no-delay and delay-aware references provide a stringent check of model implementation, extraction, and modal analysis. The variable formulations are compared with the no-delay reference; the split formulations are compared with the delayed reference. EMT Floquet modes are included to distinguish the periodically time-varying EMT result from the transformed one-step approximation.


In [ ]:
state_count = summary[
    (summary.study == "time_step_sweep") & np.isclose(summary.time_step_us, 1.0)
].set_index("model").state_count.to_dict()

reference_for = {
    "analytical no delay": None,
    "analytical split delay": None,
    "EMT variable": "analytical no delay",
    "EMT split": "analytical split delay",
    "DP Ph3 variable": "analytical no delay",
    "DP Ph3 split": "analytical split delay",
}
critical_rows = []
for model in model_order:
    mode = oscillatory_mode(model)
    reference_name = reference_for[model]
    reference = oscillatory_mode(reference_name) if reference_name else None
    critical_rows.append({
        "model/reference": model, "method": modal_method(model) or "analytical",
        "matching reference": reference_name or "—", "states": state_count.get(model, np.nan),
        "damping sigma [1/s]": mode.lambda_real,
        "frequency [Hz]": mode.lambda_imag/(2*np.pi),
        "abs damping error [1/s]": abs(mode.lambda_real-reference.lambda_real) if reference is not None else np.nan,
        "abs frequency error [Hz]": abs(mode.lambda_imag-reference.lambda_imag)/(2*np.pi) if reference is not None else np.nan,
    })
for model in ["EMT variable", "EMT split"]:
    try:
        mode = oscillatory_mode(model, method="monodromy")
    except ValueError:
        continue
    reference_name = reference_for[model]
    reference = oscillatory_mode(reference_name)
    critical_rows.append({
        "model/reference": model, "method": "Floquet",
        "matching reference": reference_name, "states": state_count.get(model, np.nan),
        "damping sigma [1/s]": mode.lambda_real,
        "frequency [Hz]": mode.lambda_imag/(2*np.pi),
        "abs damping error [1/s]": abs(mode.lambda_real-reference.lambda_real),
        "abs frequency error [Hz]": abs(mode.lambda_imag-reference.lambda_imag)/(2*np.pi),
    })
critical_table = pd.DataFrame(critical_rows)
display_numeric_table(critical_table, precision=6)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))
for model in model_order:
    modes = finite_modes(rows_for(model, 1.0, method=modal_method(model)))
    physical = modes[(modes.lambda_real > -5000) & (abs(modes.lambda_imag) < 2*np.pi*5000)]
    for ax, plotted in zip(axes, [physical, physical[physical.lambda_imag > 0]]):
        ax.scatter(plotted.lambda_real, plotted.lambda_imag/(2*np.pi),
                   marker=markers[model], s=34,
                   facecolors="none" if markers[model] in ["o", "s"] else None,
                   color=colors[model], label=model)
for ax in axes:
    ax.axvline(0, color="0.3", lw=0.8); ax.grid(alpha=0.3)
    ax.set_xlabel(r"$\mathrm{Re}\{\lambda\}$ [1/s]")
axes[0].axhline(0, color="0.3", lw=0.8)
axes[0].set(title="Physical-mode overview", ylabel="frequency [Hz]")
axes[1].set(xlim=(-45, 15), ylim=(1840, 1980),
            title="Filter/grid mode", ylabel="frequency [Hz]")
axes[1].legend(fontsize=8, ncol=2)
fig.suptitle(r"Continuous-equivalent modes, $\Delta t=1\,\mu$s")
fig.tight_layout(); plt.show()

shown_models = ["analytical no delay", "EMT split", "DP Ph3 split"]
participation_by_model = pd.concat(
    {model: grouped_participation(model) for model in shown_models}, axis=1
).fillna(0.0)
shown_groups = participation_by_model.max(axis=1).nlargest(7).index
display_numeric_table(participation_by_model.loc[shown_groups], precision=3)
participation_by_model.loc[shown_groups].iloc[::-1].plot(
    kind="barh", figsize=(10.5, 4.8), color=[colors[m] for m in shown_models]
)
plt.xlabel(r"normalized grouped $|P|$")
plt.title("Participation of the approximately 1.93 kHz pair")
plt.grid(axis="x", alpha=0.3); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()


**How to read the result.** Agreement of DP variable with the no-delay reference and DP split with the delayed reference checks the native-envelope implementations. EMT one-step and Floquet rows determine whether a remaining EMT discrepancy is caused by the transformed one-step approximation or is also present in the complete periodic map. Similar grouped participation identifies the same capacitor-voltage/current interaction across formulations. The split delay can shift damping even if delay-state participation is small: participation measures state involvement, not parameter sensitivity.

## Study 2 — time-step sensitivity and error decomposition

**Question addressed.** This study separates three effects:

1. DP variable should follow the no-delay discretized reference.
2. DP split should depart from the no-delay model, but follow the delay-aware reference.
3. EMT should approach its corresponding reference for small $\Delta t$, while the discrepancy associated with stationary-frame discretization and post-extraction transformation grows with $\Delta t$.

The first figure therefore presents damping and frequency of the tracked physical mode, separated into variable and split formulations. Continuous-equivalent interpretation is highlighted only while the mode is adequately sampled.


In [ ]:
tracked = tracked_filter_grid_modes(100.0)
floquet_tracked = floquet_filter_grid_modes(tracked)
resolution_limit_us = 1e6/(10*oscillatory_mode("analytical no delay").lambda_imag/(2*np.pi))

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.0), sharex="col")
for column, formulation in enumerate(["variable", "split"]):
    selected = (["analytical no delay", "EMT variable", "DP Ph3 variable"]
                if formulation == "variable" else
                ["analytical no delay", "analytical split delay", "EMT split", "DP Ph3 split"])
    for model in selected:
        rows = tracked[tracked.model == model].sort_values("time_step_us")
        axes[0, column].semilogx(rows.time_step_us, rows.lambda_real, "o-",
                                 color=colors[model], label=model)
        axes[1, column].semilogx(rows.time_step_us, rows.frequency_hz, "o-",
                                 color=colors[model], label=model)
    emt_model = "EMT variable" if formulation == "variable" else "EMT split"
    rows = floquet_tracked[floquet_tracked.model == emt_model].sort_values("time_step_us")
    if not rows.empty:
        axes[0, column].semilogx(rows.time_step_us, rows.lambda_real, "k--",
                                 label=f"{emt_model}, Floquet")
        axes[1, column].semilogx(rows.time_step_us, rows.frequency_hz, "k--",
                                 label=f"{emt_model}, Floquet")
    for row in range(2):
        axes[row, column].axvspan(resolution_limit_us, tracked.time_step_us.max(),
                                 color="0.9", label="<10 samples/period" if row == 0 else None)
        axes[row, column].grid(alpha=0.3)
    axes[0, column].axhline(0, color="0.3", lw=0.8)
    axes[0, column].set_title(f"{formulation.capitalize()} formulation")
    axes[0, column].legend(fontsize=7)
axes[0, 0].set_ylabel(r"damping $\sigma$ [1/s]")
axes[1, 0].set_ylabel("frequency [Hz]")
for ax in axes[1]: ax.set_xlabel(r"time step [$\mu$s]")
fig.suptitle("Tracked filter/grid mode versus time step")
fig.tight_layout(); plt.show()


The next figure performs the key decomposition. The left panel compares every extracted model with its **matching** reference. The right panel compares split results with the no-delay reference and overlays the deviation predicted solely by the analytical one-step delay. Thus, a large split-to-no-delay deviation is not treated as an implementation error when it is reproduced by the delay-aware reference.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
for model in model_order[2:]:
    rows = tracked[tracked.model == model].sort_values("time_step_us")
    axes[0].loglog(rows.time_step_us, rows.z_error_matching_reference, "o-",
                   color=colors[model], label=model)
for model in ["DP Ph3 split", "EMT split"]:
    rows = tracked[tracked.model == model].sort_values("time_step_us")
    axes[1].loglog(rows.time_step_us, rows.z_deviation_no_delay, "o-",
                   color=colors[model], label=f"{model} to no-delay")
delay = tracked[tracked.model == "analytical split delay"].sort_values("time_step_us")
axes[1].loglog(delay.time_step_us, delay.predicted_delay_z, "k--",
               label="analytical delay effect")
axes[0].set(title="Residual to matching reference",
            ylabel=r"tracked-mode $|\Delta z|$")
axes[1].set(title="Observed and predicted split-delay effect",
            ylabel=r"tracked-mode $|\Delta z|$ to no-delay reference")
for ax in axes:
    ax.set_xlabel(r"time step [$\mu$s]"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()


### Discrete stability and retained global diagnostics

The spectral radius is the primary stability criterion for the executed discrete model: $\rho<1$ is stable and $\rho>1$ is unstable. EMT Floquet multiplier radii provide the periodic-system check. The following global error plots are retained as diagnostics, but they should not be interpreted as one-to-one DP validation because the DP envelope models contain additional states and sideband modes relative to the reduced $dq0$ references.


In [ ]:
sweep = summary[summary.study == "time_step_sweep"].copy()
sweep["growth_from_radius"] = np.log(sweep.spectral_radius)/sweep.time_step_s
# Calculate multiplier radii without losing the paired real/imaginary columns.
floquet_radius = eigenvalues[
    (eigenvalues.study == "time_step_sweep") & (eigenvalues.method == "monodromy")
].assign(radius=lambda x: np.hypot(x.z_real, x.z_imag)).groupby(
    ["model", "time_step_us"], as_index=False
).radius.max()
fundamental_period = 1.0/float(parameter_table.loc["frequency", "value"])
floquet_radius["growth_from_radius"] = np.log(floquet_radius.radius)/fundamental_period

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for model in model_order[2:]:
    rows = sweep[sweep.model == model].sort_values("time_step_us")
    ax.semilogx(rows.time_step_us, rows.growth_from_radius, "o-",
                color=colors[model], label=f"{model}, one step")
for model in ["EMT variable", "EMT split"]:
    rows = floquet_radius[floquet_radius.model == model].sort_values("time_step_us")
    ax.semilogx(rows.time_step_us, rows.growth_from_radius, "--", label=f"{model}, Floquet")
ax.axhline(0.0, color="0.2", lw=1.0, label="stability boundary")
ax.set(xlabel=r"time step [$\mu$s]", ylabel="dominant growth rate [1/s]",
       title="Discrete stability versus time step")
ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
for model in model_order[2:]:
    rows = sweep[sweep.model == model].sort_values("time_step_us")
    axes[0].loglog(rows.time_step_us, rows.reference_error_z, "o-", label=model)
for model in ["EMT split", "DP Ph3 split"]:
    rows = sweep[sweep.model == model].sort_values("time_step_us")
    axes[1].loglog(rows.time_step_us, rows.split_to_no_delay_deviation, "o-", label=model)
axes[0].set(title="Global directed error to matching reference",
            ylabel=r"max nearest $|\Delta z|$")
axes[1].set(title="Global split-to-no-delay diagnostic",
            ylabel=r"max nearest finite $|\Delta\lambda|$ [1/s]")
for ax in axes:
    ax.set_xlabel(r"time step [$\mu$s]"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()


**Study 2 conclusion to check.** DP variable should remain on the no-delay reference. DP split should follow the delay-aware reference even as both depart from the no-delay result. EMT should show an additional time-step-dependent residual, while Floquet analysis determines the stability of the complete periodic EMT map. At coarse steps, $z$-plane and spectral-radius results remain meaningful even when the $1.93$ kHz continuous-frequency interpretation is no longer adequately sampled.

## Study 3 — gain and time-step stability boundaries

**Why this is included.** A short gain sweep shows that the stable and unstable time-domain cases lie on opposite sides of a continuously moving mode, rather than being isolated selected points. Longer disturbed simulations then test whether extracted damping and frequency predict the nonlinear response.


In [ ]:
gain_records = []
for gain in sorted(eigenvalues.loc[eigenvalues.study == "gain_sweep", "gain_scale"].unique()):
    no_delay = oscillatory_mode("analytical no delay", study="gain_sweep", gain_scale=gain)
    delayed = oscillatory_mode("analytical split delay", study="gain_sweep", gain_scale=gain)
    for model in model_order:
        reference = delayed if "split" in model else no_delay
        candidates = finite_modes(rows_for(model, 1.0, "gain_sweep", modal_method(model), gain))
        candidates = candidates[candidates.lambda_imag > 0]
        mode = reference if model.startswith("analytical") else nearest_mode_in_lambda(candidates, reference)
        gain_records.append({"gain_scale": gain, "model": model,
                             "damping": mode.lambda_real,
                             "frequency_hz": mode.lambda_imag/(2*np.pi)})
gain_sweep = pd.DataFrame(gain_records)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), sharey=True)
for ax, formulation in zip(axes, ["variable", "split"]):
    selected = (["analytical no delay", "EMT variable", "DP Ph3 variable"]
                if formulation == "variable" else
                ["analytical split delay", "EMT split", "DP Ph3 split"])
    for model in selected:
        rows = gain_sweep[gain_sweep.model == model]
        ax.plot(rows.gain_scale, rows.damping, "o-", color=colors[model], label=model)
    ax.axhline(0, color="0.2", lw=0.8); ax.axvline(3.7, color="0.5", ls=":")
    ax.axvline(3.8, color="0.5", ls=":")
    ax.set(title=f"{formulation.capitalize()} formulation", xlabel="gain scale")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel(r"tracked-mode damping $\sigma$ [1/s]")
fig.suptitle("Gain-driven movement of the approximately 1.93 kHz mode")
fig.tight_layout(); plt.show()

study3 = summary[summary.study.isin(["stability_stable", "stability_unstable"])].copy()
study3["predicted_stable"] = study3.spectral_radius < 1.0
display(study3[["study", "model", "time_step_us", "spectral_radius",
                "max_real_lambda", "predicted_stable"]])


### Scheduled disturbance and free response

Each simulation runs undisturbed for one 50 Hz period. At $t=20$ ms, the balanced grid-voltage reference is increased by 0.1% for 0.5 ms and then restored. Consequently, the interval before the pulse checks the initialization, the pulse provides the same physical excitation to all formulations, and the response after the pulse is an unforced free response.

The gain cases run for 0.2 s after the perturbation. The 50 and 100 $\mu$s cases run for 0.5 s because the delay-induced growth rate near the crossing is much smaller. Raw power traces retain the physical waveform; a complex-demodulated envelope is added to make modal growth or decay visible despite beating.


In [ ]:
def load_time_case(case):
    path = Path(case.log_path)
    if not path.is_absolute():
        path = build_dir / path
    if not path.exists():
        raise FileNotFoundError(f"Time-domain log not found: {path}")
    data = pd.read_csv(path, skipinitialspace=True)
    if data.empty or data.shape[1] < 2:
        raise ValueError(f"Time-domain log is empty or malformed: {path}")
    time = data.iloc[:, 0].to_numpy()
    p_columns = [c for c in data.columns if c == "p_inst" or c.startswith("p_inst_")]
    if not p_columns:
        raise KeyError(f"No p_inst column in {path}; available columns: {list(data.columns)}")
    p_column = p_columns[0]
    p_reference = float(parameter_table.loc["active_power_reference", "value"])
    signal = (data[p_column].to_numpy()-p_reference)/abs(p_reference)
    finite = np.isfinite(time) & np.isfinite(signal)
    if finite.sum() < 2:
        raise ValueError(f"Time-domain log contains fewer than two finite samples: {path}")
    return time[finite], signal[finite]

def demodulated_envelope(time, signal, frequency_hz, smoothing_time=0.01):
    dt = np.median(np.diff(time))
    samples = max(5, int(round(smoothing_time/dt)))
    baseband = signal*np.exp(-2j*np.pi*frequency_hz*time)
    real = pd.Series(baseband.real).rolling(samples, center=True, min_periods=samples//2).mean()
    imag = pd.Series(baseband.imag).rolling(samples, center=True, min_periods=samples//2).mean()
    smooth = real.to_numpy()+1j*imag.to_numpy()
    phase = np.full(len(smooth), np.nan)
    valid = np.isfinite(smooth.real) & np.isfinite(smooth.imag)
    phase[valid] = np.unwrap(np.angle(smooth[valid]))
    return 2*np.abs(smooth), phase

def plot_raw_cases(ax, cases, title):
    for case in cases.itertuples():
        time, delta_p = load_time_case(case)
        ax.plot(time, delta_p, lw=0.8, label=case.model)
    if not cases.empty:
        start = float(cases.perturbation_time_s.iloc[0])
        end = start + float(cases.perturbation_duration_s.iloc[0])
        ax.axvspan(start, end, color="0.85", label="grid-voltage pulse")
    ax.axhline(0, color="0.2", ls="--", lw=0.8)
    ax.set(title=title, xlabel="time [s]", ylabel=r"$\Delta p/P_\mathrm{ref}$")
    ax.grid(alpha=0.3); ax.legend(fontsize=7)

if time_manifest.empty:
    print("No time-domain cases. Re-run the executable with --time-domain.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), sharey=True)
    for ax, case_name in zip(axes, ["stable", "unstable"]):
        cases = time_manifest[(time_manifest.stability_case == case_name)
                              & np.isclose(time_manifest.time_step_us, 1.0)]
        plot_raw_cases(ax, cases, f"gain case: {case_name}, Δt=1 µs")
    fig.suptitle("Free response after the same scheduled disturbance")
    fig.tight_layout(); plt.show()

    fit_records = []
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
    for ax, case_name in zip(axes, ["stable", "unstable"]):
        cases = time_manifest[(time_manifest.stability_case == case_name)
                              & np.isclose(time_manifest.time_step_us, 1.0)]
        study_name = f"stability_{case_name}"
        for case in cases.itertuples():
            predicted = oscillatory_mode(case.model, 1.0, study_name)
            predicted_frequency = predicted.lambda_imag/(2*np.pi)
            time, signal = load_time_case(case)
            envelope, phase = demodulated_envelope(time, signal, predicted_frequency)
            pulse_end = case.perturbation_time_s + case.perturbation_duration_s
            fit = ((time > pulse_end+0.015) & (time < time.max()-0.015)
                   & np.isfinite(envelope) & np.isfinite(phase) & (envelope > 1e-12))
            if fit.sum() < 20:
                fit_records.append({"case": case_name, "model": case.model,
                                    "extracted damping [1/s]": predicted.lambda_real,
                                    "fitted damping [1/s]": np.nan,
                                    "extracted frequency [Hz]": predicted_frequency,
                                    "fitted frequency [Hz]": np.nan,
                                    "fit status": f"skipped: only {fit.sum()} valid samples"})
                ax.semilogy(time, envelope, label=case.model)
                continue
            slope, intercept = np.polyfit(time[fit], np.log(envelope[fit]), 1)
            fitted_frequency = predicted_frequency + np.polyfit(time[fit], phase[fit], 1)[0]/(2*np.pi)
            ax.semilogy(time, envelope, label=case.model)
            ax.semilogy(time[fit], np.exp(intercept+slope*time[fit]), "k:", alpha=0.45)
            fit_records.append({"case": case_name, "model": case.model,
                                "extracted damping [1/s]": predicted.lambda_real,
                                "fitted damping [1/s]": slope,
                                "extracted frequency [Hz]": predicted_frequency,
                                "fitted frequency [Hz]": fitted_frequency,
                                "fit status": "ok"})
        ax.set(title=case_name, xlabel="time [s]", ylabel="demodulated envelope")
        ax.grid(alpha=0.3); ax.legend(fontsize=7)
    fig.suptitle("Envelope growth/decay compared with extracted modal damping")
    fig.tight_layout(); plt.show()
    display_numeric_table(pd.DataFrame(fit_records), precision=4)

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), sharey=True)
    for ax, case_name, dt_us in zip(axes, ["delay_stable", "large_step"], [50.0, 100.0]):
        cases = time_manifest[(time_manifest.stability_case == case_name)
                              & np.isclose(time_manifest.time_step_us, dt_us)]
        plot_raw_cases(ax, cases, f"gain 3.7, Δt={dt_us:g} µs")
    fig.suptitle("Time-step-induced split-model stability change")
    fig.tight_layout(); plt.show()


**How to read Study 3.** The raw traces show the same physical disturbance and reveal whether the response remains in the small-signal region. The demodulated envelopes and fitted slopes provide the quantitative test: their signs and approximate values should follow the extracted damping. The 50/100 $\mu$s comparison brackets the split-delay stability crossing; variable models provide a no-delay baseline.

## Conclusions supported by the complete case study

1. The models remain at the intended operating point before the scheduled disturbance.
2. At small $\Delta t$, variable models agree with the no-delay reference and split models with the delay-aware reference.
3. The approximately 1.93 kHz pair is consistently associated with capacitor voltage and filter/grid currents, supporting its interpretation as a fast converter/network interaction mode.
4. The split model's deviation from the no-delay result is intentional when it follows the analytical delay-aware map.
5. EMT contains an additional time-step-dependent discrepancy associated with stationary-frame discretization and the transformed one-step modal representation; Floquet analysis provides the periodic benchmark.
6. The extracted gain- and delay-induced stability changes are validated against the growth or decay measured from the disturbed nonlinear simulations.

The global spectrum diagnostics remain useful for detecting unexpected modes, but tracked corresponding modes and spectral radii carry the primary quantitative conclusions because the reduced $dq0$ references and full DP envelope models do not have identical state dimensions.
